# LoCED × MaxText v0.2.5 — 100M full-pass training on Kaggle TPU v5e-8

This is the **practical full-training path**: a ~99.05M-parameter LoCED model
(`d=512`, 4 physical layers, 8 visits, 8 routed + 1 shared expert, 32k vocab)
trained on all six `nampdn-ai` streaming datasets with MaxText Muon.

### What it does

- builds the exact LoCED encode/decode contract:
  encode = 4 FULL; decode = 4 FULL + REUSE, REUSE, REINDEX, REINDEX (8 visits);
- streams and mixes `tiny-textbooks`, `tiny-strange-textbooks`, `tiny-codes`,
  `tiny-math-textbooks`, `tiny-orca-textbooks`, and `tiny-webtext`;
- uses dataset-specific text columns and a rolling token buffer, so source
  tokens are preserved;
- runs an 8-chip data-parallel mesh (`data=8`, `fsdp=1`), global batch
  `8 × 32 × 128 = 32,768` tokens/step;
- checkpoints model + optimizer with Orbax every 10 minutes from the start;
- stops when the streams are exhausted (step cap 200,000, above the ~4.72B
  token full-pass estimate) and writes a final checkpoint.

### Before running

1. Kaggle → **Accelerator: TPU v5e-8**.
2. Kaggle → **Add-ons → Secrets** → add `HF_TOKEN` with access to the gated
   `nampdn-ai` datasets.
3. Run all. The first install/compile takes ~5–10 minutes.

The notebook never embeds or prints the token. For the private training copy,
the token is injected into the token cell only and must be rotated afterwards.


In [ ]:
# ============================================================================
# LoCED x MaxText v0.2.5 - 100M full-pass training on Kaggle TPU v5e-8
# ============================================================================
import json
import os
import shutil
import subprocess
import sys
import textwrap
import time
from pathlib import Path

# ---------------------------------------------------------------------------
# User knobs
# ---------------------------------------------------------------------------
ACCELERATOR = os.environ.get("LOCED_ACCELERATOR", "tpu")
MAXTEXT_REF = "maxtext-v0.2.5"
MAXTEXT_REPO = "https://github.com/AI-Hypercomputer/maxtext.git"
MAXTEXT_EXTRAS = "tpu"
MODEL_CONFIG = "loced/maxtext/configs/loced_100m_tpu8.yml"
TRAIN_STEPS = int(os.environ.get("LOCED_TRAIN_STEPS", "200000"))
PER_DEVICE_BATCH = int(os.environ.get("LOCED_BATCH", "32"))
SEQ_LEN = int(os.environ.get("LOCED_SEQ", "128"))
RUN_NAME = os.environ.get("LOCED_RUN_NAME", "loced-100m-tpu8-hf")
HF_DATASETS = os.environ.get(
    "LOCED_HF_DATASETS",
    "nampdn-ai/tiny-textbooks,nampdn-ai/tiny-strange-textbooks,"
    "nampdn-ai/tiny-codes,nampdn-ai/tiny-math-textbooks,"
    "nampdn-ai/tiny-orca-textbooks,nampdn-ai/tiny-webtext",
)
HF_RATIOS = os.environ.get("LOCED_HF_RATIOS", "1,1,1,1,1,1")
HF_TOKENIZER = os.environ.get(
    "LOCED_HF_TOKENIZER", "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

LOCED_REPO_URL = os.environ.get(
    "LOCED_REPO_URL", "https://github.com/offx-zinth/loced.git"
)
LOCED_REPO_REF = os.environ.get("LOCED_REPO_REF", "master")

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
WORK.mkdir(parents=True, exist_ok=True)
MAXTEXT_DIR = WORK / "maxtext-v0.2.5"
VENV_DIR = WORK / "loced-venv"
PY = VENV_DIR / "bin" / "python"
SCRIPTS = WORK / "loced_scripts"
RUNS = WORK / "loced_runs"
CACHE = WORK / "jax_cache"
HF_CACHE = WORK / "hf_cache"
CHECKPOINT_DIR = RUNS / RUN_NAME / "checkpoints"
REPORT_100M = WORK / "kaggle_tpu8_100m_report.json"
TRAIN_LOG = WORK / "hf_train_100m.log"
for _p in (SCRIPTS, RUNS, CACHE, HF_CACHE, CHECKPOINT_DIR):
    _p.mkdir(parents=True, exist_ok=True)


def _loced_parent(root: Path):
    if (root / "loced" / "maxtext" / "train.py").is_file():
        return root
    if root.name == "loced" and (root / "maxtext" / "train.py").is_file():
        return root.parent
    return None


def _find_loced_root():
    candidates = []
    if os.environ.get("LOCED_ROOT"):
        candidates.append(Path(os.environ["LOCED_ROOT"]))
    candidates += [
        WORK,
        WORK / "loced",
        Path.cwd(),
        Path.cwd() / "loced",
        Path("/kaggle/input"),
    ]
    for root in candidates:
        found = _loced_parent(root)
        if found is not None:
            return found
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for p in kaggle_input.glob("**/maxtext/train.py"):
            repo = p.parents[1]
            if repo.name == "loced" or (
                (repo / "attention.py").is_file()
                and (repo / "config.py").is_file()
            ):
                return repo.parent
        for p in kaggle_input.glob("**/loced/maxtext/train.py"):
            return p.parents[2]
    return None


# Keep a reused Kaggle working directory in sync with GitHub master.
_target = WORK / "loced"
if LOCED_REPO_URL and (_target / ".git").is_dir():
    _env = os.environ.copy()
    _env["GIT_TERMINAL_PROMPT"] = "0"
    print(f"updating existing checkout {_target} ({LOCED_REPO_REF})")
    subprocess.run(
        ["git", "-C", str(_target), "fetch", "--depth", "1",
         "origin", LOCED_REPO_REF],
        check=True, env=_env,
    )
    subprocess.run(
        ["git", "-C", str(_target), "reset", "--hard", "FETCH_HEAD"],
        check=True, env=_env,
    )

LOCED_ROOT = _find_loced_root()
if LOCED_ROOT is None and LOCED_REPO_URL:
    target = WORK / "loced"
    if not (target / ".git").is_dir():
        if target.is_dir() and any(target.iterdir()):
            print(f"{target} exists without .git; treating it as the source")
        else:
            env = os.environ.copy()
            env["GIT_TERMINAL_PROMPT"] = "0"
            print(f"cloning {LOCED_REPO_URL} ({LOCED_REPO_REF}) -> {target}")
            subprocess.run(
                ["git", "clone", "--depth", "1", "--branch", LOCED_REPO_REF,
                 LOCED_REPO_URL, str(target)],
                check=True, env=env,
            )
    LOCED_ROOT = _find_loced_root()

print(f"ACCELERATOR : {ACCELERATOR}")
print(f"WORK        : {WORK}")
print(f"LOCED_ROOT  : {LOCED_ROOT}")
print(f"MAXTEXT_DIR : {MAXTEXT_DIR}")
print(f"VENV        : {VENV_DIR}")
print(f"MODEL       : {MODEL_CONFIG}")
print(f"RUN         : {RUN_NAME}")
print(f"STEPS       : {TRAIN_STEPS}")
print(f"BATCH       : {PER_DEVICE_BATCH} x 8 chips x {SEQ_LEN} tokens")
print(f"CHECKPOINTS : {CHECKPOINT_DIR}")
if LOCED_ROOT is None:
    raise RuntimeError(textwrap.dedent("""
        LoCED source not found.
        - Upload the repository folder (containing `loced/`) to
          /kaggle/working/loced, or as a Kaggle Dataset under
          /kaggle/input/...
        - Or set LOCED_REPO_URL (and LOCED_REPO_REF) in this cell to a branch
          that contains `loced/`.
    """))
if not (LOCED_ROOT / MODEL_CONFIG).is_file():
    raise RuntimeError(f"missing model config: {LOCED_ROOT / MODEL_CONFIG}")
print("nvidia-smi:")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "-L"], check=False)
else:
    print("nvidia-smi not found (TPU runtime)")


## 1. Python 3.12 + MaxText v0.2.5 (TPU)

MaxText v0.2.5 needs Python >= 3.12, so the notebook creates an isolated
`uv` venv. The install uses MaxText's tested TPU dependency set and pins the
HF data/tokenizer stack used by `hf_train.py`.


In [ ]:
SKIP_INSTALL = os.environ.get("LOCED_SKIP_INSTALL", "0") == "1"

if SKIP_INSTALL:
    print("LOCED_SKIP_INSTALL=1: using existing venv/MaxText checkout")
else:
    if shutil.which("uv") is None:
        print("installing uv ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    UV = shutil.which("uv")
    if UV is None:
        raise RuntimeError("uv is not on PATH after pip install")

    if not PY.exists():
        print("creating Python 3.12 venv ...")
        subprocess.run(
            [UV, "venv", "--python", "3.12", "--seed", str(VENV_DIR)],
            check=True,
        )
    print(subprocess.check_output([str(PY), "--version"], text=True).strip())

    if not (MAXTEXT_DIR / "pyproject.toml").is_file():
        print(f"cloning MaxText {MAXTEXT_REF} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", MAXTEXT_REF,
             MAXTEXT_REPO, str(MAXTEXT_DIR)],
            check=True,
        )
    else:
        print("MaxText checkout already present")

    print(f"installing maxtext[{MAXTEXT_EXTRAS}] (resolution=lowest) ...")
    subprocess.run(
        [UV, "pip", "install", "--python", str(PY), "--link-mode=copy",
         "-e", f".[{MAXTEXT_EXTRAS}]", "--resolution=lowest"],
        cwd=str(MAXTEXT_DIR), check=True,
    )
    subprocess.run(
        [UV, "pip", "install", "--python", str(PY), "--link-mode=copy",
         "pytest", "pytest-timeout",
         "datasets>=2.19,<4", "transformers>=4.40,<5", "huggingface_hub>=0.23,<1"],
        check=True,
    )
    print("dependency check:")
    subprocess.run([UV, "pip", "check", "--python", str(PY)], check=False)

print(subprocess.check_output(
    [str(PY), "-c",
     "import sys, maxtext; print(sys.version); print(maxtext.__file__)"],
    text=True,
))


## 2. TPU + real-MaxText sanity check

This fails if JAX is not on a TPU or if MaxText's real utilities cannot be
imported. A CPU-stub fallback would make the training result meaningless.


In [ ]:
BASE_ENV = os.environ.copy()
BASE_ENV.update({
    "PYTHONPATH": f"{MAXTEXT_DIR / 'src'}:{LOCED_ROOT}",
    "JAX_COMPILATION_CACHE_DIR": str(CACHE),
    "TOKENIZERS_PARALLELISM": "false",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    "HF_HOME": str(HF_CACHE),
    "HF_DATASETS_CACHE": str(HF_CACHE / "datasets"),
    "HF_HUB_CACHE": str(HF_CACHE / "hub"),
    "HF_XET_CACHE": str(HF_CACHE / "xet"),
    "PYTHONDONTWRITEBYTECODE": "1",
    "TF_CPP_MIN_LOG_LEVEL": "2",
    "LOCED_ACCELERATOR": ACCELERATOR,
})


def run_script(name, code, timeout=3600, extra_env=None, check=True, cwd=None):
    path = SCRIPTS / name
    path.write_text(textwrap.dedent(code))
    env = BASE_ENV.copy()
    if extra_env:
        env.update({str(k): str(v) for k, v in extra_env.items()})
    t0 = time.time()
    proc = subprocess.run(
        [str(PY), str(path)],
        cwd=str(cwd or LOCED_ROOT),
        env=env,
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    elapsed = time.time() - t0
    print(proc.stdout)
    if proc.stderr.strip():
        print("[stderr]\n" + proc.stderr[-12000:])
    print(f"[{name}] exit={proc.returncode} elapsed={elapsed:.1f}s")
    if check and proc.returncode != 0:
        raise RuntimeError(f"{name} failed (exit {proc.returncode})")
    return proc


def run_stream(name, code, timeout=43200, extra_env=None, log_path=None):
    """Run a long subprocess while echoing its output and tee-ing a log."""
    path = SCRIPTS / name
    path.write_text(textwrap.dedent(code))
    env = BASE_ENV.copy()
    env["PYTHONUNBUFFERED"] = "1"
    if extra_env:
        env.update({str(k): str(v) for k, v in extra_env.items()})
    log = open(log_path, "w") if log_path else None
    t0 = time.time()
    proc = subprocess.Popen(
        [str(PY), str(path)],
        cwd=str(LOCED_ROOT),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
            if log:
                log.write(line)
                log.flush()
            if time.time() - t0 > timeout:
                proc.kill()
                raise TimeoutError(f"{name} exceeded {timeout}s")
        rc = proc.wait()
    finally:
        if log:
            log.close()
    elapsed = time.time() - t0
    print(f"[{name}] exit={rc} elapsed={elapsed:.1f}s")
    if rc != 0:
        raise RuntimeError(f"{name} failed (exit {rc})")
    return rc


tpu_check = r"""
import importlib.metadata
import os
import sys

import jax
import maxtext

print("python  :", sys.version.split()[0])
print("jax     :", jax.__version__)
print("maxtext :", importlib.metadata.version("maxtext"))
print("backend :", jax.default_backend())
print("devices :", jax.devices())
assert jax.default_backend() == "tpu", (
    f"Expected TPU, got {jax.default_backend()}. Select TPU v5e-8."
)
assert len(jax.devices()) == 8, f"Expected 8 TPU chips, got {len(jax.devices())}"
for d in jax.devices():
    stats = d.memory_stats() or {}
    print(f"  {d}: {d.device_kind} bytes_limit={stats.get('bytes_limit')}")

import maxtext.utils.maxtext_utils  # noqa: F401
import maxtext.utils.model_creation_utils  # noqa: F401
print("real MaxText utilities: OK")
"""
run_script("tpu_check.py", tpu_check, timeout=900)


## 3. Hugging Face token + gated-dataset access

The token is read from the `HF_TOKEN` Kaggle secret. It is passed only through
subprocess environments and is never printed.


In [ ]:
try:
    from kaggle_secrets import UserSecretClient

    HF_TOKEN = UserSecretClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Add it in Kaggle Add-ons -> Secrets, or set the "
        "environment variable in the private training copy."
    )

os.environ["HF_TOKEN"] = HF_TOKEN
print("HF token loaded (value not printed).")

whoami_code = r"""
import os
from huggingface_hub import whoami
me = whoami(token=os.environ["HF_TOKEN"])
print("HF user:", me.get("name"))
print("HF token: OK")
"""
run_script(
    "hf_whoami.py",
    whoami_code,
    timeout=600,
    extra_env={"HF_TOKEN": HF_TOKEN},
)


## 4. Verify all six dataset schemas

The `nampdn-ai` datasets do not share one schema. This checks the exact
`textbook` / `prompt` / `response` / `human` fields before the long run starts.


In [ ]:
data_check = r"""
import os

from datasets import load_dataset
from loced.maxtext.hf_train import DEFAULT_DATASETS, _extract_text, _text_column

token = os.environ["HF_TOKEN"]
for name in DEFAULT_DATASETS:
    ds = load_dataset(
        name, split="train", streaming=True, token=token,
        trust_remote_code=True,
    )
    column = _text_column(ds.features)
    sample = None
    for row in ds:
        sample = _extract_text(name, row, ds.features)
        if sample:
            break
    if not sample:
        raise RuntimeError(f"no usable text extracted from {name}")
    print(f"{name:32s} column={column!r:24s} chars={len(sample)}")
print("all six dataset schemas: OK")
"""
run_script(
    "hf_data_check.py",
    data_check,
    timeout=2400,
    extra_env={"HF_TOKEN": HF_TOKEN},
)


## 5. 100M parameter + one-step TPU check

Builds `loced-100m` through MaxText's real `pyconfig`, checks the parameter
budget (**99,054,096** by construction), and runs two MaxText Muon updates with
the real 8-chip data-parallel batch. This catches a config, sharding or memory
problem before the multi-hour run.

The current LoCED core computes in float32, so this is the memory worst case;
it is not a bf16 throughput benchmark.


In [ ]:
bench_code = r"""
import json
import os
import time

import jax
import numpy as np
from jax.sharding import NamedSharding, PartitionSpec as P
from maxtext.configs import pyconfig

from loced.maxtext.config import LoCEDMaxTextConfig, loced_config_from_maxtext
from loced.maxtext.train import (
    create_model_and_mesh, create_optimizer, synthetic_batches, train_step,
)

config = pyconfig.initialize(
    [
        "loced-100m-bench",
        os.environ["LOCED_MODEL_CONFIG"],
        "steps=2",
        "enable_checkpointing=false",
        "run_name=loced-100m-tpu8-bench",
        "skip_jax_distributed_system=true",
    ],
    config_class=LoCEDMaxTextConfig,
)
cfg = loced_config_from_maxtext(config)
print("shape   :", f"d={cfg.hidden_size} layers={cfg.num_layers} "
      f"experts={cfg.num_experts} I={cfg.intermediate_size} "
      f"vocab={cfg.vocab_size} seq={cfg.max_seq_len}")

model, mesh = create_model_and_mesh(config)
print("mesh    :", dict(mesh.shape))
params = model.count_params()
print("params  :", f"{params:,} ({params / 1e6:.3f}M)")
assert 98_000_000 <= params <= 100_000_000, (
    f"Expected the ~99M LoCED-100M budget, got {params:,}"
)

schedule, optimizer = create_optimizer(config, model, mesh)
global_batch = int(config.per_device_batch_size) * int(config.ici_data_parallelism)
tokens = next(
    iter(synthetic_batches(cfg.vocab_size, cfg.max_seq_len, global_batch, 1, seed=0))
)
tokens = jax.device_put(tokens, NamedSharding(mesh, P("data", None)))
print("batch   :", tokens.shape, "global tokens/step:", tokens.size)

t0 = time.perf_counter()
loss0, metrics0 = train_step(model, optimizer, tokens)
jax.block_until_ready(loss0)
first_s = time.perf_counter() - t0

t0 = time.perf_counter()
loss1, metrics1 = train_step(model, optimizer, tokens)
jax.block_until_ready(loss1)
second_s = time.perf_counter() - t0

step = int(optimizer.step.get_value())
assert np.isfinite(float(loss0)) and np.isfinite(float(loss1)), (
    f"non-finite loss: {float(loss0)} -> {float(loss1)}"
)
assert step == 2, f"Expected 2 MaxText Muon updates, got {step}"

memory = {}
for device in jax.devices():
    stats = device.memory_stats() or {}
    memory[str(device)] = {
        "peak_bytes_in_use": stats.get("peak_bytes_in_use"),
        "bytes_limit": stats.get("bytes_limit"),
    }

report = {
    "backend": jax.default_backend(),
    "devices": len(jax.devices()),
    "params": params,
    "shape": {
        "hidden_size": cfg.hidden_size,
        "num_layers": cfg.num_layers,
        "num_visits": cfg.num_visits,
        "num_experts": cfg.num_experts,
        "intermediate_size": cfg.intermediate_size,
        "vocab_size": cfg.vocab_size,
        "seq_len": cfg.max_seq_len,
    },
    "mesh": {str(k): int(v) for k, v in mesh.shape.items()},
    "global_batch": int(tokens.shape[0]),
    "first_step_s": first_s,
    "second_step_s": second_s,
    "loss_first": float(loss0),
    "loss_second": float(loss1),
    "optimizer_step": step,
    "memory": memory,
}
print("first step (compile):", f"{first_s:.2f}s")
print("second step        :", f"{second_s:.2f}s")
print("loss               :", f"{float(loss0):.6f} -> {float(loss1):.6f}")
print("optimizer step     :", report["optimizer_step"])
for name, stats in memory.items():
    print("memory             :", name, stats)
with open(os.environ["LOCED_REPORT_100M"], "w") as f:
    json.dump(report, f, indent=2)
print("wrote", os.environ["LOCED_REPORT_100M"])
"""
run_script(
    "benchmark_100m.py",
    bench_code,
    timeout=3600,
    extra_env={
        "LOCED_MODEL_CONFIG": MODEL_CONFIG,
        "LOCED_REPORT_100M": str(REPORT_100M),
    },
)


## 6. Full mixed-pass training

`hf_train.py` streams all six datasets, mixes them with a shuffled equal
schedule, and trains with MaxText Muon. It stops when the streams are
exhausted (or at the 200k-step safety cap) and writes a final Orbax checkpoint.

The 10-minute wall-clock checkpoint timer starts immediately
(`LOCED_CKPT_AFTER_S=0`), so a Kaggle session interruption leaves a restorable
model + optimizer state in `CHECKPOINT_DIR`.


In [ ]:
train_code = f"""
import os

from loced.maxtext.hf_train import main

main([
    "loced-100m-tpu8-hf",
    "{MODEL_CONFIG}",
    "steps={TRAIN_STEPS}",
    "run_name={RUN_NAME}",
    "checkpoint_dir={CHECKPOINT_DIR}",
])
"""
run_stream(
    "hf_train_100m.py",
    train_code,
    timeout=43200,
    log_path=str(TRAIN_LOG),
    extra_env={
        "HF_TOKEN": HF_TOKEN,
        "HF_DATASETS": HF_DATASETS,
        "HF_RATIOS": HF_RATIOS,
        "HF_TOKENIZER": HF_TOKENIZER,
        "HF_LOG_EVERY": "100",
        "LOCED_CKPT_AFTER_S": "0",
        "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    },
)


## 7. Final checkpoint / report

The full-pass run is complete when the training subprocess exits. The final
Orbax checkpoint contains model and optimizer state and is resumable by
re-running this notebook with the same `RUN_NAME`.


In [ ]:
print("checkpoint dir:", CHECKPOINT_DIR)
ckpt_path = Path(CHECKPOINT_DIR)
files = sorted(p for p in ckpt_path.rglob("*") if p.is_file()) if ckpt_path.is_dir() else []
size = sum(p.stat().st_size for p in files)
print("checkpoint files:", len(files), "bytes:", size)
print("latest checkpoint entries:", [str(p.relative_to(ckpt_path)) for p in files[:20]])

if TRAIN_LOG.is_file():
    lines = TRAIN_LOG.read_text(errors="replace").splitlines()
    print("last training log lines:")
    print("\n".join(lines[-40:]))
if REPORT_100M.is_file():
    print("100M check report:")
    print(REPORT_100M.read_text())
